In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from keras.optimizers import Adam
from keras.losses import Loss
from joblib import Parallel, delayed
from scipy.optimize import minimize
from tensorflow.keras.initializers import GlorotUniform, GlorotNormal, HeUniform
import keras.backend as K
import statsmodels.api as sm

In [ ]:
# Define DPD custom loss function as a class
class DPDLoss(Loss):
    def __init__(self, sigma, alpha):
        super().__init__()
        self.alpha = alpha
        self.sigma = float(sigma)

    def call(self, y_true, y_pred):
        diff = (y_true - y_pred)/self.sigma
        diff_sq = tf.clip_by_value(diff**2, 1e-10, 1e6)  # Prevents extreme values
        normal_dist = tf.exp(-0.5 * diff_sq)
        normal_dist = tf.clip_by_value(normal_dist, 1e-10, 1.0)  # Keep in safe range
        loss = 1/((self.sigma**self.alpha)*((1+self.alpha)**0.5)) - (1+1/self.alpha)*tf.reduce_mean((normal_dist/self.sigma)**self.alpha)
        # Debugging
        #tf.print("Loss:", loss)
        #loss = tf.debugging.check_numerics(loss, "NaN detected in loss function!")
        return loss

def trimmed_mean(arr, trim_fraction):
    arr_sorted = np.sort(arr)  # Step 1: Sort array
    trim_count = int(len(arr) * trim_fraction)  # Step 2: Compute number of elements to trim
    
    if trim_count == 0:  # Ensure at least one element is trimmed if possible
        return np.mean(arr)
    
    trimmed_arr = arr_sorted[:-trim_count]  # Step 3: Remove largest 20%
    return np.mean(trimmed_arr)  # Step 4: Compute mean of remaining elements

def H(sigma, alpha, diff):
    normal_dist = np.exp(-0.5 * (diff / sigma) ** 2) / sigma
    loss = 1 / ((sigma ** alpha) * np.sqrt(1 + alpha)) - (1 + 1 / alpha) * np.mean(normal_dist ** alpha)
    return loss

def sig_hat_MAD(resi):
    return 1.4826 * np.median(np.abs(resi - np.median(resi)))

In [ ]:
df = pd.read_csv('boston_mdev_scaled.csv')
df.shape

In [ ]:
# print(df.columns)

In [ ]:
X_w = df.drop(['medv'], axis=1)
Y_w = df['medv']
n = X_w.shape[0]
X_w.shape, Y_w.shape, n

In [ ]:
def QQ_Resi(Y, Y_pred, font = 15):
    _, axes = plt.subplots(2, 1, figsize=(5,8))
    resi = Y - Y_pred
    axes[0].scatter(Y_pred, resi, s = 15)  # Fixed ax issue
    axes[0].set_title('Residuals vs Fitted')
    axes[0].set_xlabel('Fitted Values', fontsize = font)
    axes[0].set_ylabel('Residuals', fontsize=font)
    axes[0].tick_params(axis='both', labelsize=font)
    sns.histplot(resi, bins=20, kde=True, ax=axes[1])
    axes[1].set_xlabel('Residuals', fontsize=font)
    axes[1].set_title('Histogram of residuals with KDE', fontsize=font)
    axes[1].tick_params(axis='both', labelsize=font)
    plt.tight_layout()  # Adjust layout to prevent overlap
    plt.show()

# MLE

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

model = keras.Sequential([
    keras.layers.Dense(8, activation='relu', kernel_initializer=GlorotUniform(seed=42)),
    keras.layers.Dense(1, kernel_initializer=GlorotUniform(seed=42))
    ])

model.compile(optimizer='adam', loss='mse')
model.fit(X_w, Y_w, epochs=3000, verbose = 0)

mle_pred = model.predict(X_w, verbose = 0).flatten()

In [ ]:
QQ_Resi(Y_w, mle_pred)

# DPD

In [ ]:
diff = Y_w - mle_pred; abs_diff = np.abs(diff)
sig0 = sig_hat_MAD(diff)
sig0

### $\alpha = 0.3$

In [ ]:
num_iterations = 20
np.random.seed(42)
tf.random.set_seed(42)
model = keras.Sequential([
    keras.layers.Dense(8, activation='relu', kernel_initializer=GlorotUniform(seed=42)),
    keras.layers.Dense(1, kernel_initializer=GlorotUniform(seed=42))
    ])

# Initial value of sigma
sig0 = 0.058  # Set your initial sigma value
alpha = 0.3

for i in range(num_iterations):
    model.compile(optimizer=Adam(learning_rate=0.001), loss=DPDLoss(sigma = sig0, alpha=alpha))
    if(i>0):
        model.set_weights(weights)
    model.fit(X_w, Y_w, epochs=200, verbose = 0)
    dpd_pred = model.predict(X_w, verbose=0).flatten()
    weights = model.get_weights()  #  print(weights)
    diff = np.array(Y_w - dpd_pred)
    sig0 = minimize(H, sig0, args = (alpha,diff), method='L-BFGS-B', bounds = [(0.001,10)]).x[0]

dpd_pred = model.predict(X_w, verbose=0).flatten()

In [ ]:
QQ_Resi(Y_w, dpd_pred)

### $\alpha = 0.5$

In [ ]:
num_iterations = 20
np.random.seed(42)
tf.random.set_seed(42)
model = keras.Sequential([
    keras.layers.Dense(8, activation='relu', kernel_initializer=GlorotUniform(seed=42)),
    keras.layers.Dense(1, kernel_initializer=GlorotUniform(seed=42))
    ])

# Initial value of sigma
sig0 = 0.058  # Set your initial sigma value
alpha = 0.5

for i in range(num_iterations):
    model.compile(optimizer=Adam(learning_rate=0.001), loss=DPDLoss(sigma = sig0, alpha=alpha))
    if(i>0):
        model.set_weights(weights)
    model.fit(X_w, Y_w, epochs=200, verbose = 0)
    dpd_pred = model.predict(X_w, verbose=0).flatten()
    weights = model.get_weights()  #  print(weights)
    diff = np.array(Y_w - dpd_pred)
    sig0 = minimize(H, sig0, args = (alpha,diff), method='L-BFGS-B', bounds = [(0.001,10)]).x[0]

dpd_pred = model.predict(X_w, verbose=0).flatten()


In [ ]:
QQ_Resi(Y_w, dpd_pred)